```
lgbm pipeline/
├── src/                                   # ソースコード
│   ├── 00_preprocessing_{カテゴリ名}.ipynb
│   ├── 01_eda_{カテゴリ名}.ipynb
│   ├── 02_feature_engineering_{カテゴリ名}.ipynb
│   ├── 03_train_lgbm_{カテゴリ名}.ipynb
│   ├── 04_evaluation_{カテゴリ名}.ipynb
│   └── 05_prediction_lgbm_{カテゴリ名}.ipynb
│
├── data/                                  # データディレクトリ
│   ├── raw/                               # 元データ
│   │
│   ├── processed/                         # 前処理済みデータ
│   │
│   ├── abt/                               # 分析用テーブル（ABT）
│   │   ├── abt_train.csv
│   │   └── abt_test.csv
│   │
│   └── output/                            # 中間出力データ
│
│
└── output/                                # 分析・評価結果
    ├── YYMMDDHHMM
    │   ├── model/                         # モデル関連ファイル
    │   ├── params/                        # パラメータ管理
    │   └── output/
    │       └── output files
    └──...
    


```


# モデル学習(LightGBM)

LightGBMを利用したモデル学習を実施するフェーズです。

## 実施内容
* ベースラインモデル学習
* ハイパーパラメータチューニング
* チューニング済みモデル学習
* パラメータ比較
* モデル予測
* MLflowによる実験管理

## 1. ライブラリインポート

In [75]:
# 標準ライブラリ
import json
import logging
import os
import warnings
from functools import wraps
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

# サードパーティライブラリ
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from tqdm.auto import tqdm

# 機械学習
import lightgbm as lgb

# ハイパーパラメータ最適化 (HPO)
from ray import tune
from ray.tune.schedulers import ASHAScheduler
from ray.tune.search.optuna import OptunaSearch
from autogluon.tabular import TabularPredictor

# モデル管理
import mlflow

# ONNX
from onnxmltools.convert import convert_lightgbm
from onnxmltools.convert.common.data_types import FloatTensorType

# 自作モジュール
from utils_copy import (
    timing_decorator,
    get_timestamp,
    create_output_dir,
    load_data_pd,
    save_csvs,
    split_data_kfold,
    setup_mlflow,
    run_hpo,
    export_onnx_lgbm,
)
load_dotenv()


# ロガー設定
# python規約9.4のlogging_config.pyを参考に、loggerを設定
# INFO以上(INFO, WARNING, ERROR, CRITICAL)を出力
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

warnings.filterwarnings('default')

logger.info("ライブラリのインポート完了")


2026-06-11 06:37:29,064 - INFO - ライブラリのインポート完了


## 2. パス設定
* 入力・出力ファイルのパスを定義する
* タイムスタンプ付きの出力ディレクトリを作成
* モデル・パラメータを保存する

In [76]:
# ==共通化== →utils.py
# timestampの取得
from datetime import datetime
def get_timestamp() -> str:
    """出力ディレクトリやファイル名に利用するタイムスタンプ文字列を返す。"""
    return datetime.now().strftime("%y%m%d%H%M")

# 出力ディレクトリの作成
@timing_decorator
def create_output_dir(base_dir: str = "../output", timestamp: Optional[str] = None) -> Path:
    output_timestamp: str = timestamp or get_timestamp()
    output_dir = Path(base_dir) / output_timestamp
    output_dir.mkdir(parents=True, exist_ok=True)
    subdirs = ["model", "params", "artifacts"]
    for sub in subdirs:
        (output_dir / sub).mkdir(parents=True, exist_ok=True)
    logger.info("出力ディレクトリを作成しました: %s", output_dir)
    return output_dir

In [119]:
# パスをローカルではなく、blobのパスを指定するようにする

# ディレクトリ
DATA_DIR: str = "../data"
OUTPUT_TIMESTAMP: str = get_timestamp()
OUTPUT_DIR: Path = create_output_dir(timestamp=OUTPUT_TIMESTAMP)
MODEL_DIR: str = f"{OUTPUT_DIR}/model"
PARAMS_DIR: str = f"{OUTPUT_DIR}/params"

# 入力
TRAIN_DATA_PATH: str = "../data/train.csv"


# 出力
# 分割済みデータ
X_TRAIN_FILE: str = f"{DATA_DIR}/X_train.csv"
X_VALID_FILE: str = f"{DATA_DIR}/X_valid.csv"
Y_TRAIN_FILE: str = f"{DATA_DIR}/y_train.csv"
Y_VALID_FILE: str = f"{DATA_DIR}/y_valid.csv"

# ベースラインモデル
MODEL_BASELINE_FILE: str = f"{MODEL_DIR}/model_baseline_{OUTPUT_TIMESTAMP}.lgbm"
MODEL_BASELINE_ONNX: str = f"{MODEL_DIR}/model_baseline_{OUTPUT_TIMESTAMP}.onnx"
PARAMS_BASELINE_FILE: str = f"{PARAMS_DIR}/params_baseline_{OUTPUT_TIMESTAMP}.json"

# チューニング済みモデル
MODEL_TUNED_FILE: str = f"{MODEL_DIR}/model_tuned_{OUTPUT_TIMESTAMP}.lgbm"
MODEL_TUNED_ONNX: str = f"{MODEL_DIR}/model_tuned_{OUTPUT_TIMESTAMP}.onnx"
PARAMS_TUNED_FILE: str = f"{PARAMS_DIR}/params_tuned_{OUTPUT_TIMESTAMP}.json"

# チューニング結果
HPO_TUNING_RESULT_FILE: str = f"{PARAMS_DIR}/hpo_tuning_result_{OUTPUT_TIMESTAMP}.json"
AUTOML_TUNING_RESULT_FILE: str = f"{PARAMS_DIR}/automl_tuning_result_{OUTPUT_TIMESTAMP}.json"
# パラメータ比較結果
PARAMS_COMPARISON_FILE: str = f"{PARAMS_DIR}/params_comparison_{OUTPUT_TIMESTAMP}.csv"

# 予測結果
PREDICTION_RESULT_FILE: str = f"{OUTPUT_DIR}/prediction_result_{OUTPUT_TIMESTAMP}.csv"

2026-06-11 07:13:55,979 - INFO - 出力ディレクトリを作成しました: ../output/2606110713
2026-06-11 07:13:55,981 - INFO - create_output_dir 実行時間: 0.00秒


2026-06-11 07:14:33,925 - INFO - Request URL: 'https://westus2.livediagnostics.monitor.azure.com/QuickPulseService.svc/ping?api-version=2024-04-01-preview&ikey=REDACTED'
Request method: 'POST'
Request headers:
    'Content-Type': 'application/json'
    'Content-Length': '308'
    'x-ms-qps-transmission-time': 'REDACTED'
    'x-ms-qps-machine-name': 'REDACTED'
    'x-ms-qps-instance-name': 'REDACTED'
    'x-ms-qps-stream-id': 'REDACTED'
    'x-ms-qps-role-name': 'REDACTED'
    'x-ms-qps-invariant-version': 'REDACTED'
    'x-ms-qps-configuration-etag': 'REDACTED'
    'Accept': 'application/json'
A body is sent with the request
2026-06-11 07:14:34,052 - INFO - Response status: 200
Response headers:
    'Server': 'Microsoft-IIS/10.0'
    'x-ms-qps-subscribed': 'REDACTED'
    'Request-Context': 'REDACTED'
    'Access-Control-Expose-Headers': 'REDACTED'
    'X-Powered-By': 'REDACTED'
    'Strict-Transport-Security': 'REDACTED'
    'X-Content-Type-Options': 'REDACTED'
    'Date': 'Wed, 10 Jun

## 3. 変数定義
* 学習に使用する変数を定義
* モデルパラメータを定義

In [78]:
# ターゲットカラム名
TARGET_COLUMN: str = "y"

# RandomState
RANDOM_STATE: int = 42

# 学習データとテストデータの分割比率
TEST_SIZE: float = 0.2

# k-分割cross-validationでの分割数
kfold_split: int = 5

# データ読み込み時のチャンクサイズ
CHUNK_SIZE = 1000

In [79]:
# ベースラインで使用するパラメータ
PARAMS_BASELINE: Dict[str, Any] = {
    'objective': 'binary',
    'metric': 'auc',
    'max_depth': 5,
    'learning_rate': 0.1,
    'n_estimators': 100,
    'random_state': RANDOM_STATE,
    'verbose': 0 
}

EARLY_STOPPING_ROUNDS: int = 100
LOG_EVALUATION_PERIOD: int = 100

In [80]:
# チューニング後のパラメータの叩き台
params_tuned: Dict[str, Any] = {
    'objective': 'binary',
    'metric': 'auc',
    'max_depth': 7,
    'learning_rate': 0.05,
    'n_estimators': 1000,
    'num_leaves': 64,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': RANDOM_STATE,
    'verbose': 0  
}

tuning_early_stopping_rounds: int = 50

## 4. MLflowセットアップ
* Azure MLワークスペースに接続
* MLflowの実験管理環境を設定

In [ ]:
SUBSCRIPTION_ID: str = os.getenv("SUBSCRIPTION_ID")
RESOURCE_GROUP: str = os.getenv("RESOURCE_GROUP")
WORKSPACE_NAME: str = os.getenv("WORKSPACE_NAME")
experiment_name: str = "banking_lgbm"
registered_model_name: str = "banking_lgbm_model"
artifact_path: str = "model"
datastore_name: str = "metric_params_dev"
datastore_account_name: str = "nbadevshuei8224694400"
container_name: str = "azureml"

In [ ]:
# ==共通化== →utils.py
# MLセットアップ
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.ai.ml.entities import AzureBlobDatastore

@timing_decorator
def setup_mlflow(
    subscription_id: str,
    resource_group: str,
    workspace_name: str,
    experiment_name: str,
    datastore_name: str,
    datastore_account_name: str,
    container_name: str,
) -> None:
    ml_client: MLClient = MLClient(
        DefaultAzureCredential(),
        subscription_id,
        resource_group,
        workspace_name,
    )
    
    tracking_uri: str = (
        ml_client.workspaces
        .get(workspace_name)
        .mlflow_tracking_uri
    )
    mlflow.set_tracking_uri(tracking_uri)
    mlflow.set_experiment(experiment_name)

    store = AzureBlobDatastore(
    name=datastore_name,
    account_name=datastore_account_name,
    container_name=container_name
)
    ml_client.create_or_update(store)
    
    
    logger.info("MLflow tracking URI設定完了")
    logger.info("Experiment: %s", experiment_name)

In [ ]:
setup_mlflow(
    subscription_id=SUBSCRIPTION_ID,
    resource_group=RESOURCE_GROUP,
    workspace_name=WORKSPACE_NAME,
    experiment_name=experiment_name,
    datastore_name=datastore_name,
    datastore_account_name=datastore_account_name,
    container_name=container_name,
)

2026-06-11 06:37:48,226 - INFO - No environment configuration found.
2026-06-11 06:37:48,228 - INFO - ManagedIdentityCredential will use IMDS
2026-06-11 06:37:48,445 - WARNING - Overriding of current MeterProvider is not allowed
2026-06-11 06:37:48,447 - WARNING - Overriding of current TracerProvider is not allowed
2026-06-11 06:37:48,450 - WARNING - Overriding of current LoggerProvider is not allowed
2026-06-11 06:37:48,451 - WARNING - /Users/aa539999/.pyenv/versions/3.10.18/lib/python3.10/site-packages/azure/monitor/opentelemetry/_configure.py:275: DeprecationWarning: You should use `LoggerProvider` instead. Deprecated since version 1.39.0 and will be removed in a future release.
  event_provider = EventLoggerProvider(logger_provider)

2026-06-11 06:37:48,459 - WARNING - Attempting to instrument while already instrumented
2026-06-11 06:37:48,463 - WARNING - Attempting to instrument while already instrumented
2026-06-11 06:37:48,466 - WARNING - Attempting to instrument while already i

## 5. データ処理
* 学習データの読み込み
* K-Fold分割
* カテゴリ・数値変数の定義

### 5.1 データ読み込み

In [84]:
# ==共通化== →utils.py
# データの読み込み（チャンク化して進捗バー表示）
@timing_decorator
def load_data_pd(path: str, chunk_size: int) -> pd.DataFrame:
    chunks = []
    for chunk in tqdm(pd.read_csv(path, chunksize=chunk_size)):
        chunks.append(chunk)
    return pd.concat(chunks, ignore_index=True)

2026-06-11 06:37:56,137 - INFO - Request URL: 'https://westus2-0.in.applicationinsights.azure.com//v2.1/track'
Request method: 'POST'
Request headers:
    'Content-Type': 'application/json'
    'Content-Length': '1325'
    'Accept': 'application/json'
    'x-ms-client-request-id': 'a4a73a12-6514-11f1-8dc0-faef0ead971b'
    'User-Agent': 'azsdk-python-monitor-opentelemetry-exporter/1.0.0b1 Python/3.10.18 (macOS-26.4-arm64-arm-64bit)'
A body is sent with the request


In [ ]:
df: pd.DataFrame = load_data_pd(path=TRAIN_DATA_PATH, chunk_size=CHUNK_SIZE)

0it [00:00, ?it/s]

2026-06-11 06:37:56,279 - INFO - load_data_pd 実行時間: 0.11秒


2026-06-11 06:37:56,344 - INFO - Response status: 200
Response headers:
    'Transfer-Encoding': 'chunked'
    'Content-Type': 'application/json; charset=utf-8'
    'Server': 'Microsoft-HTTPAPI/2.0'
    'Strict-Transport-Security': 'REDACTED'
    'X-Content-Type-Options': 'REDACTED'
    'Date': 'Wed, 10 Jun 2026 21:37:56 GMT'
2026-06-11 06:37:56,345 - INFO - Transmission succeeded: Item received: 1. Items accepted: 1
2026-06-11 06:37:58,334 - INFO - Request URL: 'https://westus2-0.in.applicationinsights.azure.com//v2.1/track'
Request method: 'POST'
Request headers:
    'Content-Type': 'application/json'
    'Content-Length': '2416'
    'Accept': 'application/json'
    'x-ms-client-request-id': 'a5f68422-6514-11f1-8dc0-faef0ead971b'
    'User-Agent': 'azsdk-python-monitor-opentelemetry-exporter/1.0.0b1 Python/3.10.18 (macOS-26.4-arm64-arm-64bit)'
A body is sent with the request
2026-06-11 06:37:58,523 - INFO - Response status: 200
Response headers:
    'Transfer-Encoding': 'chunked'
   

### 5.2 データ分割

In [86]:
# ==共通化== →utils.py
# データ分割（Stratified K-Fold）
from sklearn.model_selection import StratifiedKFold


@timing_decorator
def split_data_kfold(
    df: pd.DataFrame,
    target: str,
    n_splits: int = 5,
    random_state: int = 42
) -> List[Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]]:
    feature_df = df.drop(columns=[target])
    target_values = df[target].values
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    folds = []
    for fold_idx, (train_idx, valid_idx) in enumerate(skf.split(feature_df, target_values), 1):
        x_train = feature_df.iloc[train_idx].reset_index(drop=True)
        x_valid = feature_df.iloc[valid_idx].reset_index(drop=True)
        y_train = pd.DataFrame({target: target_values[train_idx]})
        y_valid = pd.DataFrame({target: target_values[valid_idx]})
        logger.info("Fold %s: 訓練=%s, 検証=%s", fold_idx, len(x_train), len(x_valid))
        folds.append((x_train, x_valid, y_train, y_valid))
    return folds

In [87]:
# K-Fold分割（全てのfoldを取得）
folds = split_data_kfold(
    df=df,
    target=TARGET_COLUMN,
    n_splits=kfold_split,
    random_state=RANDOM_STATE
)

# 最初のfoldを取得（データ保存用）→Loopしてすべてのfoldを学習するように修正
X_train, X_valid, y_train, y_valid = folds[0]

2026-06-11 06:38:02,132 - INFO - Fold 1: 訓練=21702, 検証=5426
2026-06-11 06:38:02,140 - INFO - Fold 2: 訓練=21702, 検証=5426
2026-06-11 06:38:02,149 - INFO - Fold 3: 訓練=21702, 検証=5426
2026-06-11 06:38:02,157 - INFO - Fold 4: 訓練=21703, 検証=5425
2026-06-11 06:38:02,166 - INFO - Fold 5: 訓練=21703, 検証=5425
2026-06-11 06:38:02,167 - INFO - split_data_kfold 実行時間: 0.06秒


### 5.3 データ保存

In [88]:
# ==共通化== →utils.py
@timing_decorator
def save_csvs(files: Dict[str, pd.DataFrame]) -> None:
    for file_path, df in files.items():
        df.to_csv(file_path, index=False)
        logger.info("%s 保存完了", file_path)

In [89]:
save_csvs(
    {
        X_TRAIN_FILE: X_train,
        X_VALID_FILE: X_valid,
        Y_TRAIN_FILE: y_train,
        Y_VALID_FILE: y_valid,
    }
)

2026-06-11 06:38:06,676 - INFO - ../data/X_train.csv 保存完了
2026-06-11 06:38:06,692 - INFO - ../data/X_valid.csv 保存完了
2026-06-11 06:38:06,699 - INFO - ../data/y_train.csv 保存完了
2026-06-11 06:38:06,703 - INFO - ../data/y_valid.csv 保存完了
2026-06-11 06:38:06,703 - INFO - save_csvs 実行時間: 0.10秒


### 5.4 カテゴリ・数値変数定義

In [90]:
from dataclasses import dataclass
from typing import List

@dataclass
class FeatureSchema:
    target: str
    num_cols: List[str]
    cat_cols: List[str]

In [91]:
schema = FeatureSchema(
    target="label",
    num_cols=['id', 'age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous'],
    cat_cols= ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']   
)

## 6. モデル学習
* モデル学習関数を定義
* ベースラインモデルを学習

### 6.1 モデル学習用データ準備
* Lightgbmモデル学習用のデータセットを準備

In [92]:
def to_categorical(df, cat_cols):
    for col in cat_cols:
        df[col] = df[col].astype("category")
    return df

In [ ]:
@timing_decorator
def prepare_lgbm_data(
    X_train: pd.DataFrame,
    X_valid: pd.DataFrame,
    y_train: pd.DataFrame,
    y_valid: pd.DataFrame,
    target_column: str,
    cat_cols: list[str],
) -> tuple[lgb.Dataset, lgb.Dataset]:

    X_train_pd = X_train.copy()
    X_valid_pd = X_valid.copy()

    y_train_pd = y_train[target_column] if isinstance(y_train, pd.DataFrame) else y_train
    y_valid_pd = y_valid[target_column] if isinstance(y_valid, pd.DataFrame) else y_valid

    X_train_pd = to_categorical(X_train_pd, cat_cols)
    X_valid_pd = to_categorical(X_valid_pd, cat_cols)

        
    lgb_train = lgb.Dataset(
        X_train_pd,
        label=y_train_pd,
    )

    lgb_valid = lgb.Dataset(
        X_valid_pd,
        label=y_valid_pd,
        reference=lgb_train,
    )

    return lgb_train, lgb_valid

### 6.2 学習関数定義
* モデル学習関数を定義

In [94]:
def train_lgbm(
    lgb_train: lgb.Dataset,
    lgb_valid: lgb.Dataset,
    params: Dict[str, Any],
    early_stopping_rounds: int,
    log_evaluation_period: int,
) -> lgb.Booster:

    return lgb.train(
        params=params,
        train_set=lgb_train,
        valid_sets=[lgb_train, lgb_valid],
        valid_names=["train", "valid"],
        callbacks=[
            lgb.early_stopping(early_stopping_rounds),
            lgb.log_evaluation(log_evaluation_period),
        ],
    )

In [ ]:
def train_lgbm_kfold(
    folds,
    params: Dict[str, Any],
) -> list[lgb.Booster]:

    models = []

    for i, (X_train, X_valid, y_train, y_valid) in enumerate(folds):

        logger.info("Training fold %d", i)

        lgb_train, lgb_valid = prepare_lgbm_data(
            X_train=X_train,
            X_valid=X_valid,
            y_train=y_train,
            y_valid=y_valid,
            target_column=TARGET_COLUMN,
            cat_cols=schema.cat_cols,
        )

        model = train_lgbm(
            lgb_train=lgb_train,
            lgb_valid=lgb_valid,
            params=params,
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            log_evaluation_period=LOG_EVALUATION_PERIOD,
        )

        models.append(model)

    return models

### 6.3 MLflowログ
* MLflowへのメトリクス・パラメータ記録関数を定義

In [95]:
def log_training_result(
    model: lgb.Booster,
    params: Dict[str, Any],
    model_name: str,
    early_stopping_rounds: int,
) -> None:

    mlflow.log_params(params)
    mlflow.log_param("model_type", model_name)
    mlflow.log_param(
        "early_stopping_rounds",
        early_stopping_rounds,
    )

    mlflow.log_metric(
        "best_iteration",
        model.best_iteration,
    )

    mlflow.log_metric(
        "train_auc",
        model.best_score["train"]["auc"],
    )

    mlflow.log_metric(
        "valid_auc",
        model.best_score["valid"]["auc"],
    )


### 6.4 出力モデルを.onnxで保存
* 出力モデルを.onnxで保存し、mlflowへ登録

In [96]:
# ==LGMB関連関数==
# モデルファイルをonnxに変換
def export_onnx_lgbm(
    model: lgb.Booster,
    train_pd: pd.DataFrame,
    onnx_path: str,
    mlflow_artifact_path: str = "model/onnx"
) -> None:
    n_features = train_pd.shape[1]
    initial_type = [("input", FloatTensorType([None, n_features]))]
    onnx_model = convert_lightgbm(model, initial_types=initial_type)
    with open(onnx_path, "wb") as f:
        f.write(onnx_model.SerializeToString())
    mlflow.log_artifact(onnx_path, artifact_path=mlflow_artifact_path)


### 6.5 ベースラインモデル学習
* ベースラインパラメータを使用してモデルを学習

【実施後、コメントアウトします】

In [97]:
lgb_train, lgb_valid = prepare_lgbm_data(
    X_train=X_train,
    X_valid=X_valid,
    y_train=y_train,
    y_valid=y_valid,
    target_column=TARGET_COLUMN,
    cat_cols=schema.cat_cols,
)


2026-06-11 06:38:24,690 - INFO - prepare_lgbm_data 実行時間: 0.02秒


2026-06-11 06:38:24,854 - INFO - Request URL: 'https://westus2.livediagnostics.monitor.azure.com/QuickPulseService.svc/ping?api-version=2024-04-01-preview&ikey=REDACTED'
Request method: 'POST'
Request headers:
    'Content-Type': 'application/json'
    'Content-Length': '308'
    'x-ms-qps-transmission-time': 'REDACTED'
    'x-ms-qps-machine-name': 'REDACTED'
    'x-ms-qps-instance-name': 'REDACTED'
    'x-ms-qps-stream-id': 'REDACTED'
    'x-ms-qps-role-name': 'REDACTED'
    'x-ms-qps-invariant-version': 'REDACTED'
    'x-ms-qps-configuration-etag': 'REDACTED'
    'Accept': 'application/json'
A body is sent with the request
2026-06-11 06:38:25,007 - INFO - Response status: 200
Response headers:
    'Server': 'Microsoft-IIS/10.0'
    'x-ms-qps-subscribed': 'REDACTED'
    'Request-Context': 'REDACTED'
    'Access-Control-Expose-Headers': 'REDACTED'
    'X-Powered-By': 'REDACTED'
    'Strict-Transport-Security': 'REDACTED'
    'X-Content-Type-Options': 'REDACTED'
    'Date': 'Wed, 10 Jun

In [ ]:
with mlflow.start_run(run_name="baseline_lgbm"):

    model_baseline = train_lgbm(  
        lgb_train=lgb_train,
        lgb_valid=lgb_valid,
        params=PARAMS_BASELINE,
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        log_evaluation_period=LOG_EVALUATION_PERIOD,
    )

    log_training_result(
        model=model_baseline,
        params=PARAMS_BASELINE,
        model_name="baseline_lgbm",
        early_stopping_rounds=100,
    )

[LightGBM] [Warning] Provided parameters constrain tree depth (max_depth=5) without explicitly setting 'num_leaves'. This can lead to underfitting. To resolve this warning, pass 'num_leaves' (<=32) in params. Alternatively, pass (max_depth=-1) and just use 'num_leaves' to constrain model complexity.
[LightGBM] [Warning] Provided parameters constrain tree depth (max_depth=5) without explicitly setting 'num_leaves'. This can lead to underfitting. To resolve this warning, pass 'num_leaves' (<=32) in params. Alternatively, pass (max_depth=-1) and just use 'num_leaves' to constrain model complexity.
[LightGBM] [Warning] Provided parameters constrain tree depth (max_depth=5) without explicitly setting 'num_leaves'. This can lead to underfitting. To resolve this warning, pass 'num_leaves' (<=32) in params. Alternatively, pass (max_depth=-1) and just use 'num_leaves' to constrain model complexity.
Training until validation scores don't improve for 100 rounds
[LightGBM] [Warning] No further spl

In [102]:
# モデル・パラメータファイルを保存
model_baseline.save_model(MODEL_BASELINE_FILE)
with open(PARAMS_BASELINE_FILE, "w") as f:
    json.dump(PARAMS_BASELINE, f, indent=2)

logger.info("ベースラインモデル保存: %s", MODEL_BASELINE_FILE)
logger.info("ベーxスラインパラメータ保存: %s", PARAMS_BASELINE_FILE)


2026-06-11 06:40:01,121 - INFO - ベースラインモデル保存: ../output/2606110637/model/model_baseline_2606110637.lgbm
2026-06-11 06:40:01,122 - INFO - ベーxスラインパラメータ保存: ../output/2606110637/model/params_baseline_2606110637.json


In [103]:
# ベースラインモデルをONNX形式で保存する
export_onnx_lgbm(
    model=model_baseline,
    train_pd=X_train,
    onnx_path=MODEL_BASELINE_ONNX
)
logger.info("ONNXモデル保存: %s", MODEL_BASELINE_ONNX)


2026-06-11 06:40:03,230 - WARNING - /Users/aa539999/.pyenv/versions/3.10.18/lib/python3.10/site-packages/mlflow/store/tracking/rest_store.py:211: DeprecationWarning: label() is deprecated. Use is_required() or is_repeated() instead.
  req_body = message_to_json(

2026-06-11 06:40:03,855 - INFO - AzureCliCredential.get_token succeeded
2026-06-11 06:40:03,856 - INFO - DefaultAzureCredential acquired a token from AzureCliCredential


MlflowException: API request to endpoint /api/2.0/mlflow/runs/create failed with error code 403 != 200. Response body: ''

2026-06-11 06:40:08,926 - INFO - Request URL: 'https://westus2-0.in.applicationinsights.azure.com//v2.1/track'
Request method: 'POST'
Request headers:
    'Content-Type': 'application/json'
    'Content-Length': '2354'
    'Accept': 'application/json'
    'x-ms-client-request-id': 'f3cd3e84-6514-11f1-8dc0-faef0ead971b'
    'User-Agent': 'azsdk-python-monitor-opentelemetry-exporter/1.0.0b1 Python/3.10.18 (macOS-26.4-arm64-arm-64bit)'
A body is sent with the request
2026-06-11 06:40:09,236 - INFO - Response status: 200
Response headers:
    'Transfer-Encoding': 'chunked'
    'Content-Type': 'application/json; charset=utf-8'
    'Server': 'Microsoft-HTTPAPI/2.0'
    'Strict-Transport-Security': 'REDACTED'
    'X-Content-Type-Options': 'REDACTED'
    'Date': 'Wed, 10 Jun 2026 21:40:08 GMT'
2026-06-11 06:40:09,237 - INFO - Transmission succeeded: Item received: 2. Items accepted: 2
2026-06-11 06:40:25,287 - INFO - Request URL: 'https://westus2.livediagnostics.monitor.azure.com/QuickPulseSer

## 7. ハイパーパラメータチューニング(コード修正中)

* Optuna+Ray Tune
* AutoGluon

### Optuna + Ray Tune

In [ ]:
# Ray Tune から呼び出されるトレーニング関数
def run_lgbm_trial(config: Dict[str, Any]) -> None:

    lgb_params: Dict[str, Any] = {**PARAMS_BASELINE, **config}
    
    # KFoldの中で平均評価する場合
    X_train, X_valid, y_train, y_valid = folds[0]

    # prepare_lgbm_data で lgb.Dataset を生成
    lgb_train_hpo, lgb_valid_hpo = prepare_lgbm_data(
        X_train=X_train,
        X_valid=X_valid,
        y_train=y_train,
        y_valid=y_valid,
        target_column=TARGET_COLUMN,
        cat_cols=schema.cat_cols,
    )
    # free_raw_data=False にしておくと Ray ワーカーが Dataset を再構築できる
    lgb_train_hpo.free_raw_data = False
    lgb_valid_hpo.free_raw_data = False

    model: lgb.Booster = train_lgbm(
        lgb_train=lgb_train_hpo,
        lgb_valid=lgb_valid_hpo,
        params=lgb_params,
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        log_evaluation_period=LOG_EVALUATION_PERIOD,
    )

    tune.report({
        "auc": model.best_score["valid"]["auc"],
        "best_iteration": model.best_iteration,
    })

In [115]:
@timing_decorator
def run_hpo(
    trainable,
    param_space: Dict[str, Any],
    metric: str,
    mode: str,
    num_samples: int,
):
    search_alg = OptunaSearch(metric=metric, mode=mode)
    tuner = tune.Tuner(
        trainable,
        tune_config=tune.TuneConfig(
            metric=metric,
            mode=mode,
            scheduler=ASHAScheduler(),
            search_alg=search_alg,
            num_samples=num_samples,
        ),
        param_space=param_space,
    )
    results = tuner.fit()
    best_result = results.get_best_result(metric=metric, mode=mode)
    logger.info("Best config: %s", best_result.config)
    return results, best_result

In [ ]:
param_space: Dict[str, Any] = {
    "num_leaves":    tune.randint(16, 256),
    "max_depth":     tune.randint(3, 10),
    "learning_rate": tune.loguniform(0.01, 0.2),
}

results, best_result = run_hpo(
    trainable=run_lgbm_trial,
    param_space=param_space,
    metric="auc",
    mode="max",
    num_samples=2,
)

(run_lgbm_trial pid=19248) [LightGBM] [Warning] No further splits with positive gain, best gain: -inf
(run_lgbm_trial pid=19248) Training until validation scores don't improve for 100 rounds
(run_lgbm_trial pid=19248) [LightGBM] [Warning] No further splits with positive gain, best gain: -inf
(run_lgbm_trial pid=19248) [LightGBM] [Warning] No further splits with positive gain, best gain: -inf
(run_lgbm_trial pid=19248) [LightGBM] [Warning] No further splits with positive gain, best gain: -inf
(run_lgbm_trial pid=19248) [LightGBM] [Warning] No further splits with positive gain, best gain: -inf
(run_lgbm_trial pid=19248) [LightGBM] [Warning] No further splits with positive gain, best gain: -inf
(run_lgbm_trial pid=19248) [LightGBM] [Warning] No further splits with positive gain, best gain: -inf
(run_lgbm_trial pid=19248) [LightGBM] [Warning] No further splits with positive gain, best gain: -inf
(run_lgbm_trial pid=19248) [LightGBM] [Warning] No further splits with positive gain, best gain

2026-06-11 07:10:20,786	INFO tune.py:1009 -- Wrote the latest version of all result files and experiment state to '/Users/aa539999/ray_results/run_lgbm_trial_2026-06-11_07-10-12' in 0.0055s.
2026-06-11 07:10:20,791	INFO tune.py:1041 -- Total run time: 8.05 seconds (8.01 seconds for the tuning loop).
2026-06-11 07:10:20,796 - INFO - Best config: {'num_leaves': 46, 'max_depth': 8, 'learning_rate': 0.02000772018818124}
2026-06-11 07:10:20,798 - INFO - run_hpo 実行時間: 8.10秒
2026-06-11 07:10:20,799 - INFO - Best HPO config: {'num_leaves': 46, 'max_depth': 8, 'learning_rate': 0.02000772018818124}


2026-06-11 07:10:32,975 - INFO - Request URL: 'https://westus2.livediagnostics.monitor.azure.com/QuickPulseService.svc/ping?api-version=2024-04-01-preview&ikey=REDACTED'
Request method: 'POST'
Request headers:
    'Content-Type': 'application/json'
    'Content-Length': '308'
    'x-ms-qps-transmission-time': 'REDACTED'
    'x-ms-qps-machine-name': 'REDACTED'
    'x-ms-qps-instance-name': 'REDACTED'
    'x-ms-qps-stream-id': 'REDACTED'
    'x-ms-qps-role-name': 'REDACTED'
    'x-ms-qps-invariant-version': 'REDACTED'
    'x-ms-qps-configuration-etag': 'REDACTED'
    'Accept': 'application/json'
A body is sent with the request
2026-06-11 07:10:34,092 - INFO - Response status: 503
Response headers:
    'Content-Type': 'text/html'
    'Server': 'Microsoft-IIS/10.0'
    'X-Powered-By': 'REDACTED'
    'Strict-Transport-Security': 'REDACTED'
    'X-Content-Type-Options': 'REDACTED'
    'Date': 'Wed, 10 Jun 2026 22:10:33 GMT'
    'Content-Length': '27'
2026-06-11 07:10:34,093 - WARNING - Ran i

In [ ]:
# hpo params update
best_config = best_result.config
hpo_params_tuned = params_tuned.update(best_config)
# hpo params save
with open(HPO_TUNING_RESULT_FILE, "w") as f:
    json.dump(hpo_params_tuned, f, indent=2)


### Autogluon

In [ ]:
train_data = X_train.copy()
train_data["target"] = y_train

valid_data = X_valid.copy()
valid_data["target"] = y_valid

In [65]:
predictor = TabularPredictor(
    label="target",
    eval_metric="roc_auc",
).fit(
    train_data=train_data,
    hyperparameters={
        "GBM": params_tuned
    },
    presets="high_quality",
    time_limit=100,
)
logger.info("Models: %s", predictor.model_names())


No path specified. Models will be saved in: "AutogluonModels/ag-20260610_133111"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.10.18
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.4.0: Thu Mar 19 19:30:44 PDT 2026; root:xnu-12377.101.15~1/RELEASE_ARM64_T6000
CPU Count:          10
Pytorch Version:    2.9.1
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       2.88 GB / 16.00 GB (18.0%)
Disk Space Avail:   132.53 GB / 460.43 GB (28.8%)
Presets specified: ['high_quality']
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
Note: `save_bag_folds=False`! This will greatly reduce peak

(_ray_fit pid=45833) [LightGBM] [Warning] No further splits with positive gain, best gain: -inf


	0.9287	 = Validation score   (roc_auc)
	2.34s	 = Training   runtime
	0.47s	 = Validation runtime
Fitting model: WeightedEnsemble_L2 ... Training model for up to 24.84s of the 19.39s of remaining time.
	Fitting 1 model on all data | Fitting with cpus=10, gpus=0, mem=0.0/3.1 GB
	Ensemble Weights: {'LightGBM_BAG_L1': 1.0}
	0.9287	 = Validation score   (roc_auc)
	0.05s	 = Training   runtime
	0.0s	 = Validation runtime
Fitting 1 L2 models, fit_strategy="sequential" ...
Fitting model: LightGBM_BAG_L2 ... Training model for up to 19.27s of the 19.27s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with ParallelLocalFoldFittingStrategy (8 workers, per: cpus=1, gpus=0, memory=1.02%)


(_ray_fit pid=51990) [LightGBM] [Warning] No further splits with positive gain, best gain: -inf [repeated 663x across cluster]


	0.9276	 = Validation score   (roc_auc)
	2.67s	 = Training   runtime
	0.2s	 = Validation runtime
Fitting model: WeightedEnsemble_L3 ... Training model for up to 24.84s of the 13.89s of remaining time.
	Fitting 1 model on all data | Fitting with cpus=10, gpus=0, mem=0.0/2.7 GB
	Ensemble Weights: {'LightGBM_BAG_L1': 0.611, 'LightGBM_BAG_L2': 0.389}
	0.9291	 = Validation score   (roc_auc)
	0.12s	 = Training   runtime
	0.0s	 = Validation runtime
AutoGluon training complete, total runtime = 11.27s ... Best model: WeightedEnsemble_L3 | Estimated inference throughput: 3575.8 rows/s (2412 batch size)
Automatically performing refit_full as a post-fit operation (due to `.fit(..., refit_full=True)`
Refitting models via `predictor.refit_full` using all of the data (combined train and validation)...
	Models trained in this way will have the suffix "_FULL" and have NaN validation score.
	This process is not bound by time_limit, but should take less time than the original `predictor.fit` call.
	To le

...

(_ray_fit pid=52175) [LightGBM] [Warning] No further splits with positive gain, best gain: -inf [repeated 236x across cluster]


	0.9306	 = Validation score   (roc_auc)
	2.98s	 = Training   runtime
	0.71s	 = Validation runtime
Fitting model: WeightedEnsemble_L2 ... Training model for up to 70.79s of the 65.10s of remaining time.
	Fitting 1 model on all data | Fitting with cpus=10, gpus=0, mem=0.0/2.6 GB
	Ensemble Weights: {'LightGBM_BAG_L1': 1.0}
	0.9306	 = Validation score   (roc_auc)
	0.05s	 = Training   runtime
	0.01s	 = Validation runtime
AutoGluon training complete, total runtime = 6.01s ... Best model: WeightedEnsemble_L2 | Estimated inference throughput: 3804.9 rows/s (2713 batch size)
Automatically performing refit_full as a post-fit operation (due to `.fit(..., refit_full=True)`
Refitting models via `predictor.refit_full` using all of the data (combined train and validation)...
	Models trained in this way will have the suffix "_FULL" and have NaN validation score.
	This process is not bound by time_limit, but should take less time than the original `predictor.fit` call.
	To learn more, refer to the `.re

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

	8.81s	 = Training   runtime
Fitting model: WeightedEnsemble_L2_FULL | Skipping fit via cloning parent ...
	Ensemble Weights: {'LightGBM_BAG_L1': 1.0}
	0.05s	 = Training   runtime
Updated best model to "LightGBM_BAG_L1_FULL" (Previously "WeightedEnsemble_L2"). AutoGluon will default to using "LightGBM_BAG_L1_FULL" for predict() and predict_proba().
Refit complete, total runtime = 8.89s ... Best model: "LightGBM_BAG_L1_FULL"
TabularPredictor saved. To load, use: predictor = TabularPredictor.load("/Users/aa539999/Desktop/pipeline/src/AutogluonModels/ag-20260610_133111")
2026-06-10 22:31:55,549 - INFO - Models: ['LightGBM_BAG_L1', 'WeightedEnsemble_L2', 'LightGBM_BAG_L1_FULL', 'WeightedEnsemble_L2_FULL']


2026-06-10 22:32:12,065 - INFO - Request URL: 'https://westus2.livediagnostics.monitor.azure.com/QuickPulseService.svc/ping?api-version=2024-04-01-preview&ikey=REDACTED'
Request method: 'POST'
Request headers:
    'Content-Type': 'application/json'
    'Content-Length': '308'
    'x-ms-qps-transmission-time': 'REDACTED'
    'x-ms-qps-machine-name': 'REDACTED'
    'x-ms-qps-instance-name': 'REDACTED'
    'x-ms-qps-stream-id': 'REDACTED'
    'x-ms-qps-role-name': 'REDACTED'
    'x-ms-qps-invariant-version': 'REDACTED'
    'x-ms-qps-configuration-etag': 'REDACTED'
    'Accept': 'application/json'
A body is sent with the request
2026-06-10 22:32:12,244 - INFO - Response status: 200
Response headers:
    'Server': 'Microsoft-IIS/10.0'
    'x-ms-qps-subscribed': 'REDACTED'
    'Request-Context': 'REDACTED'
    'Access-Control-Expose-Headers': 'REDACTED'
    'X-Powered-By': 'REDACTED'
    'Strict-Transport-Security': 'REDACTED'
    'X-Content-Type-Options': 'REDACTED'
    'Date': 'Wed, 10 Jun

In [ ]:
# AutoGluonが学習した全モデル名のうち、LightGBMモデル抽出
lgb_models = [
    model_name
    for model_name in predictor.model_names()
    if "LightGBM" in model_name
]
if len(lgb_models) == 0:
    raise ValueError("LightGBM model not found")

lgb_model_name = lgb_models[0]

logger.info("load model: %s", lgb_model_name)
model = predictor._trainer.load_model(lgb_model_name)


2026-06-10 22:33:07,013 - INFO - load model: LightGBM_BAG_L1


In [ ]:
# autogluon params update
autogluon_params_tuned = params_tuned.update(model.params)

with open(AUTOML_TUNING_RESULT_FILE, "w") as f:
    json.dump(autogluon_params_tuned, f, indent=2)


2026-06-10 22:33:10,900 - INFO - AutoGluon LightGBM params: {'use_orig_features': True, 'valid_stacker': True, 'max_base_models': 0, 'max_base_models_per_type': 'auto', 'save_bag_folds': False, 'stratify': 'auto', 'bin': 'auto', 'n_bins': None, 'vary_seed_across_folds': False, 'model_random_seed': 0}
2026-06-10 22:33:10,901 - INFO - Updated tuned params: {'objective': 'binary', 'metric': 'auc', 'max_depth': 7, 'learning_rate': 0.02427941843702629, 'n_estimators': 1000, 'num_leaves': 45, 'min_child_samples': 20, 'subsample': 0.8, 'colsample_bytree': 0.8, 'random_state': 42, 'verbose': 0, 'use_orig_features': True, 'valid_stacker': True, 'max_base_models': 0, 'max_base_models_per_type': 'auto', 'save_bag_folds': False, 'stratify': 'auto', 'bin': 'auto', 'n_bins': None, 'vary_seed_across_folds': False, 'model_random_seed': 0}


2026-06-10 22:33:12,307 - INFO - Request URL: 'https://westus2.livediagnostics.monitor.azure.com/QuickPulseService.svc/ping?api-version=2024-04-01-preview&ikey=REDACTED'
Request method: 'POST'
Request headers:
    'Content-Type': 'application/json'
    'Content-Length': '308'
    'x-ms-qps-transmission-time': 'REDACTED'
    'x-ms-qps-machine-name': 'REDACTED'
    'x-ms-qps-instance-name': 'REDACTED'
    'x-ms-qps-stream-id': 'REDACTED'
    'x-ms-qps-role-name': 'REDACTED'
    'x-ms-qps-invariant-version': 'REDACTED'
    'x-ms-qps-configuration-etag': 'REDACTED'
    'Accept': 'application/json'
A body is sent with the request
2026-06-10 22:33:12,455 - INFO - Response status: 200
Response headers:
    'Server': 'Microsoft-IIS/10.0'
    'x-ms-qps-subscribed': 'REDACTED'
    'Request-Context': 'REDACTED'
    'Access-Control-Expose-Headers': 'REDACTED'
    'X-Powered-By': 'REDACTED'
    'Strict-Transport-Security': 'REDACTED'
    'X-Content-Type-Options': 'REDACTED'
    'Date': 'Wed, 10 Jun

## 8. チューニング済みモデル学習
* チューニング済みパラメータでモデルを再学習
* ベースラインモデルとの性能比較を行う

In [ ]:
# tunedパラメータでLightGBMモデルを学習する
mlflow.end_run()
with mlflow.start_run(run_name="tuned_lgbm"):

    model_tuned = train_lgbm(
        lgb_train=lgb_train,
        lgb_valid=lgb_valid,
        params=params_tuned,
        early_stopping_rounds=tuning_early_stopping_rounds,
        log_evaluation_period=LOG_EVALUATION_PERIOD,
    )

    log_training_result(
        model=model_tuned,
        params=params_tuned, #hpo_params_tunedまたはautogluon_params_tunedを指定
        model_name="tuned_lgbm",
        early_stopping_rounds=tuning_early_stopping_rounds,
    )

2026-06-10 19:41:54,339 - WARNING - /Users/aa539999/.pyenv/versions/3.10.18/lib/python3.10/site-packages/mlflow/store/tracking/rest_store.py:211: DeprecationWarning: label() is deprecated. Use is_required() or is_repeated() instead.
  req_body = message_to_json(

2026-06-10 19:41:54,764 - INFO - AzureCliCredential.get_token succeeded
2026-06-10 19:41:54,765 - INFO - DefaultAzureCredential acquired a token from AzureCliCredential


[LightGBM] [Warning] Provided parameters constrain tree depth (max_depth=5) without explicitly setting 'num_leaves'. This can lead to underfitting. To resolve this warning, pass 'num_leaves' (<=32) in params. Alternatively, pass (max_depth=-1) and just use 'num_leaves' to constrain model complexity.
[LightGBM] [Warning] Provided parameters constrain tree depth (max_depth=5) without explicitly setting 'num_leaves'. This can lead to underfitting. To resolve this warning, pass 'num_leaves' (<=32) in params. Alternatively, pass (max_depth=-1) and just use 'num_leaves' to constrain model complexity.
Training until validation scores don't improve for 50 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


2026-06-10 19:41:55,441 - INFO - Request URL: 'https://westus2-0.in.applicationinsights.azure.com//v2.1/track'
Request method: 'POST'
Request headers:
    'Content-Type': 'application/json'
    'Content-Length': '3902'
    'Accept': 'application/json'
    'x-ms-client-request-id': 'ffd9a840-64b8-11f1-8dc0-faef0ead971b'
    'User-Agent': 'azsdk-python-monitor-opentelemetry-exporter/1.0.0b1 Python/3.10.18 (macOS-26.4-arm64-arm-64bit)'
A body is sent with the request


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


2026-06-10 19:41:55,742 - INFO - Response status: 200
Response headers:
    'Transfer-Encoding': 'chunked'
    'Content-Type': 'application/json; charset=utf-8'
    'Server': 'Microsoft-HTTPAPI/2.0'
    'Strict-Transport-Security': 'REDACTED'
    'X-Content-Type-Options': 'REDACTED'
    'Date': 'Wed, 10 Jun 2026 10:41:55 GMT'
2026-06-10 19:41:55,743 - INFO - Transmission succeeded: Item received: 3. Items accepted: 3


...

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Early stopping, best iteration is:
[100]	train's auc: 0.969231	valid's auc: 0.933134


2026-06-10 19:41:57,354 - INFO - AzureCliCredential.get_token succeeded
2026-06-10 19:41:57,355 - INFO - DefaultAzureCredential acquired a token from AzureCliCredential
2026-06-10 19:41:57,979 - INFO - AzureCliCredential.get_token succeeded
2026-06-10 19:41:57,980 - INFO - DefaultAzureCredential acquired a token from AzureCliCredential
2026-06-10 19:41:58,492 - INFO - AzureCliCredential.get_token succeeded
2026-06-10 19:41:58,493 - INFO - DefaultAzureCredential acquired a token from AzureCliCredential
2026-06-10 19:41:58,628 - WARNING - /Users/aa539999/.pyenv/versions/3.10.18/lib/python3.10/site-packages/mlflow/store/tracking/rest_store.py:516: DeprecationWarning: label() is deprecated. Use is_required() or is_repeated() instead.
  req_body = message_to_json(

2026-06-10 19:41:59,005 - INFO - AzureCliCredential.get_token succeeded
2026-06-10 19:41:59,006 - INFO - DefaultAzureCredential acquired a token from AzureCliCredential
2026-06-10 19:41:59,418 - INFO - Request URL: 'https://westu

🏃 View run tuned_lgbm at: https://japaneast.api.azureml.ms/mlflow/v2.0/subscriptions/741fd3f6-d4fa-49d7-a9a5-ebcb4d75f141/resourceGroups/nba_dev_shuei_rg/providers/Microsoft.MachineLearningServices/workspaces/nba_dev_shuei/#/experiments/4d781a1c-d8fb-4066-9f7a-40170671ea47/runs/6a95187a-0759-48be-aa33-87d58318724b
🧪 View experiment at: https://japaneast.api.azureml.ms/mlflow/v2.0/subscriptions/741fd3f6-d4fa-49d7-a9a5-ebcb4d75f141/resourceGroups/nba_dev_shuei_rg/providers/Microsoft.MachineLearningServices/workspaces/nba_dev_shuei/#/experiments/4d781a1c-d8fb-4066-9f7a-40170671ea47


2026-06-10 19:42:02,283 - INFO - AzureCliCredential.get_token succeeded
2026-06-10 19:42:02,284 - INFO - DefaultAzureCredential acquired a token from AzureCliCredential


2026-06-10 19:42:06,005 - INFO - Request URL: 'https://westus2-0.in.applicationinsights.azure.com//v2.1/track'
Request method: 'POST'
Request headers:
    'Content-Type': 'application/json'
    'Content-Length': '5542'
    'Accept': 'application/json'
    'x-ms-client-request-id': '06258f66-64b9-11f1-8dc0-faef0ead971b'
    'User-Agent': 'azsdk-python-monitor-opentelemetry-exporter/1.0.0b1 Python/3.10.18 (macOS-26.4-arm64-arm-64bit)'
A body is sent with the request
2026-06-10 19:42:06,495 - INFO - Response status: 200
Response headers:
    'Transfer-Encoding': 'chunked'
    'Content-Type': 'application/json; charset=utf-8'
    'Server': 'Microsoft-HTTPAPI/2.0'
    'Strict-Transport-Security': 'REDACTED'
    'X-Content-Type-Options': 'REDACTED'
    'Date': 'Wed, 10 Jun 2026 10:42:05 GMT'
2026-06-10 19:42:06,497 - INFO - Transmission succeeded: Item received: 4. Items accepted: 4


In [ ]:
model_tuned.save_model(MODEL_TUNED_FILE)
logger.info("モデル保存: %s", MODEL_TUNED_FILE)

2026-06-10 19:47:54,505 - INFO - モデル保存: ../output/2606101938/model/model_tuned_2606101938.lgbm
2026-06-10 19:47:54,506 - INFO - パラメータ保存: ../output/2606101938/model/params_tuned_2606101938.json


2026-06-10 19:48:01,238 - INFO - Request URL: 'https://westus2-0.in.applicationinsights.azure.com//v2.1/track'
Request method: 'POST'
Request headers:
    'Content-Type': 'application/json'
    'Content-Length': '5177'
    'Accept': 'application/json'
    'x-ms-client-request-id': 'd9e1d800-64b9-11f1-8dc0-faef0ead971b'
    'User-Agent': 'azsdk-python-monitor-opentelemetry-exporter/1.0.0b1 Python/3.10.18 (macOS-26.4-arm64-arm-64bit)'
A body is sent with the request
2026-06-10 19:48:01,433 - INFO - Response status: 200
Response headers:
    'Transfer-Encoding': 'chunked'
    'Content-Type': 'application/json; charset=utf-8'
    'Server': 'Microsoft-HTTPAPI/2.0'
    'Strict-Transport-Security': 'REDACTED'
    'X-Content-Type-Options': 'REDACTED'
    'Date': 'Wed, 10 Jun 2026 10:48:00 GMT'
2026-06-10 19:48:01,434 - INFO - Transmission succeeded: Item received: 8. Items accepted: 8
2026-06-10 19:48:01,563 - INFO - Request URL: 'https://westus2.livediagnostics.monitor.azure.com/QuickPulseSer

## 9. パラメータ比較
* ベースラインとチューニング済みのパラメータを比較
* 比較結果をCSVファイルに保存

In [30]:
@timing_decorator
def compare_parameters(
    baseline_params: Dict[str, Any],
    tuned_params: Dict[str, Any],
    comparison_file: str,
) -> pd.DataFrame:

    changed_params_df = pd.DataFrame([
        {
            "parameter": key,
            "baseline": str(baseline_params.get(key, "N/A")),
            "tuned": str(tuned_params.get(key, "N/A")),
        }
        for key in sorted(set(baseline_params) | set(tuned_params))
        if baseline_params.get(key, "N/A")
        != tuned_params.get(key, "N/A")
    ])

    changed_params_df.to_csv(comparison_file, index=False)

    logger.info("変更されたパラメータ数: %d", len(changed_params_df))
    logger.info("\n%s", changed_params_df)
    logger.info("比較結果保存: %s", comparison_file)

    return changed_params_df

In [31]:
# パラメータ比較実行
comparison_df: pd.DataFrame = compare_parameters(
    baseline_params=PARAMS_BASELINE,
    tuned_params=params_tuned,
    comparison_file=PARAMS_COMPARISON_FILE
)

2026-06-10 19:42:14,061 - INFO - 変更されたパラメータ数: 7


2026-06-10 19:42:14,067 - INFO - 
           parameter baseline tuned
0   colsample_bytree      N/A   0.8
1      learning_rate      0.1  0.05
2          max_depth        5     7
3  min_child_samples      N/A    20
4       n_estimators      100  1000
5         num_leaves      N/A    64
6          subsample      N/A   0.8
2026-06-10 19:42:14,080 - INFO - 比較結果保存: ../output/2606101938/model/params_comparison_2606101938.csv
2026-06-10 19:42:14,082 - INFO - compare_parameters 実行時間: 0.03秒


## 10. モデル予測
* ベースラインモデルとチューニング後のモデル使用して予測
* 予測結果をCSVファイルに保存

In [ ]:
def prepare_prediction_data(
    model: lgb.Booster,
    X_valid: pd.DataFrame,
    y_valid: pd.DataFrame,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:

    X_valid_pd = X_valid.copy()
    y_valid_np = y_valid.values

    to_categorical(X_valid_pd, schema.cat_cols)

    pred_proba = model.predict(X_valid_pd)
    pred = (pred_proba > 0.5).astype(int)

    return y_valid_np, pred_proba, pred

2026-06-11 07:24:36,252 - INFO - Request URL: 'https://westus2.livediagnostics.monitor.azure.com/QuickPulseService.svc/ping?api-version=2024-04-01-preview&ikey=REDACTED'
Request method: 'POST'
Request headers:
    'Content-Type': 'application/json'
    'Content-Length': '308'
    'x-ms-qps-transmission-time': 'REDACTED'
    'x-ms-qps-machine-name': 'REDACTED'
    'x-ms-qps-instance-name': 'REDACTED'
    'x-ms-qps-stream-id': 'REDACTED'
    'x-ms-qps-role-name': 'REDACTED'
    'x-ms-qps-invariant-version': 'REDACTED'
    'x-ms-qps-configuration-etag': 'REDACTED'
    'Accept': 'application/json'
A body is sent with the request
2026-06-11 07:24:36,372 - INFO - Response status: 200
Response headers:
    'Server': 'Microsoft-IIS/10.0'
    'x-ms-qps-subscribed': 'REDACTED'
    'Request-Context': 'REDACTED'
    'Access-Control-Expose-Headers': 'REDACTED'
    'X-Powered-By': 'REDACTED'
    'Strict-Transport-Security': 'REDACTED'
    'X-Content-Type-Options': 'REDACTED'
    'Date': 'Wed, 10 Jun

In [ ]:
# ベースラインモデル予測
y_valid_np, baseline_proba, baseline_pred = prepare_prediction_data(
    model=model_baseline,
    X_valid=X_valid,
    y_valid=y_valid,
)

# チューニングモデル予測
y_valid_np, tuned_proba, tuned_pred = prepare_prediction_data(
    model=model_tuned,
    X_valid=X_valid,
    y_valid=y_valid,
)

# 結果比較DataFrame
prediction_df = pd.DataFrame({
    "y": y_valid_np.ravel(),
    "y_baseline_proba": baseline_proba,
    "y_baseline_pred": baseline_pred,
    "y_tuned_proba": tuned_proba,
    "y_tuned_pred": tuned_pred,
})

prediction_df.to_csv(PREDICTION_RESULT_FILE, index=False)

logger.info("予測結果保存: %s", PREDICTION_RESULT_FILE)
logger.info("\n%s", prediction_df.head())

2026-06-11 07:24:36,542 - INFO - 予測結果保存: ../output/2606110713/prediction_result_2606110713.csv
2026-06-11 07:24:36,543 - INFO - 
   y  y_baseline_proba  y_baseline_pred  y_tuned_proba  y_tuned_pred
0  1          0.134213                0       0.065755             0
1  1          0.737685                1       0.499179             0
2  1          0.543359                1       0.674729             1
3  1          0.384004                0       0.318602             0
4  0          0.015514                0       0.012816             0


2026-06-11 07:24:53,522 - INFO - Request URL: 'https://westus2-0.in.applicationinsights.azure.com//v2.1/track'
Request method: 'POST'
Request headers:
    'Content-Type': 'application/json'
    'Content-Length': '5175'
    'Accept': 'application/json'
    'x-ms-client-request-id': '33f20638-651b-11f1-8dc0-faef0ead971b'
    'User-Agent': 'azsdk-python-monitor-opentelemetry-exporter/1.0.0b1 Python/3.10.18 (macOS-26.4-arm64-arm-64bit)'
A body is sent with the request
2026-06-11 07:24:53,753 - INFO - Response status: 200
Response headers:
    'Transfer-Encoding': 'chunked'
    'Content-Type': 'application/json; charset=utf-8'
    'Server': 'Microsoft-HTTPAPI/2.0'
    'Strict-Transport-Security': 'REDACTED'
    'X-Content-Type-Options': 'REDACTED'
    'Date': 'Wed, 10 Jun 2026 22:24:53 GMT'
2026-06-11 07:24:53,754 - INFO - Transmission succeeded: Item received: 8. Items accepted: 8
2026-06-11 07:25:36,482 - INFO - Request URL: 'https://westus2.livediagnostics.monitor.azure.com/QuickPulseSer